Carga de importaciones y modelo beto.

In [1]:
import torch
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification)
from torch.utils.data import DataLoader

RUTA_MODELO = r"C:\Users\GGNADI2\OneDrive - MAPFRE\Escritorio\Análisis_sentimiento\beto-sentiment"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(
    RUTA_MODELO,
    local_files_only=True
)

model = AutoModelForSequenceClassification.from_pretrained(
    RUTA_MODELO,
    local_files_only=True
)

model.to(device)
model.eval()

C:\Users\GGNADI2\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 339.20it/s, Materializing param=classifier.weight]                                      
BertForSequenceClassification LOAD REPORT from: C:\Users\GGNADI2\OneDrive - MAPFRE\Escritorio\Análisis_sentimiento\beto-sentiment
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.

BertForSequenceClassification LOAD REPORT from: C:\Users\GGNADI2\OneDrive - MAPFRE\Escritorio\Análisis_sentimiento\beto-sentiment
Key                          | Status     |  | 
------------

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(31006, 768, padding_idx=1)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

Carga excel

In [2]:
df = pd.read_excel("Datos/asis_completo.xlsx")
df.to_csv("Datos/asis_completo.csv", index=False, sep=",", encoding="utf-8-sig")

Limpieza y tokenización

In [3]:
import re

df = pd.read_csv("Datos/asis_completo.csv")

df = df.dropna(subset=["ES - Corredores - Verbatim Comment including Corredores"])
df["texto"] = df["ES - Corredores - Verbatim Comment including Corredores"].astype(str)

def normalizar_texto(texto):
    texto = texto.strip()
    texto = re.sub(r"\s+", " ", texto)
    texto = texto.lower()
    return texto

df["texto_norm"] = df["texto"].apply(normalizar_texto)

def tokenize_batch(textos):
    return tokenizer(
        textos,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )
batch_size = 32

loader = DataLoader(
    df["texto_norm"].tolist(),
    batch_size=batch_size
)

sentimientos = []
confianzas = []

with torch.no_grad():
    for batch in loader:
        inputs = tokenize_batch(batch)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)

        pred = torch.argmax(probs, dim=1)
        conf = probs.max(dim=1).values

        sentimientos.extend(pred.tolist())
        confianzas.extend(conf.tolist())

mapa = {
    0: "Negativo",
    1: "Positivo",
}

df["sentimiento"] = sentimientos
df["sentimiento_label"] = df["sentimiento"].map(mapa)
df["confianza"] = confianzas

C:\Users\GGNADI2\AppData\Local\Temp\ipykernel_16064\549477176.py:3: DtypeWarning: Columns (0: ES - DNI, 1: ES - BirthDate, 2: DR, 3: Export AT Description, 4: ES - Unit ID Text, 5: ES - Unit ID, 6: ES - Auto - Number Plate, 7: ES - Auto - Type management - String, 8: ES - MAPFRE Santander Record.1, 9: ES - Contact Intrest including Corredores, 10: ¿Has conseguido contactar con el cliente?, 11: Indique el motivo de insatisfacción detectado en el asegurado, 12: Selecciona el motivo de insatisfacción del asegurado (dentro del, 13: Solicitud del servicio de asistencia, 14: ¿El cliente queda conforme?, 15: Indique la propuesta ofrecida al asegurado, 16: Si ha seleccionado 'Otros', por favor especifique cuál) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("Datos/asis_completo.csv")


Análisis de sentimientos

In [4]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

RUTA_EMBEDDINGS = r"C:\Users\GGNADI2\OneDrive - MAPFRE\Escritorio\Análisis_sentimiento\beto-sentiment"

tokenizer_emb = AutoTokenizer.from_pretrained(
    RUTA_EMBEDDINGS,
    local_files_only=True
)

model_emb = AutoModel.from_pretrained(
    RUTA_EMBEDDINGS,
    local_files_only=True
)

model_emb.eval()

def obtener_embedding(texto):
    texto = normalizar_texto(texto)

    inputs = tokenizer_emb(
        texto,
        truncation=True,
        padding=True,
        max_length=256,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = model_emb(**inputs)

    embedding = outputs.last_hidden_state[:, 0, :].numpy()
    return embedding


def normalizar(v):
    return v / np.linalg.norm(v)

proto_positivo = [
    "la atención fue adecuada",
    "el servicio ha sido satisfactorio",
    "todo se resolvió correctamente",
    "buena atención recibida",
    "gestión correcta del caso"
]

proto_neutro = [
    "se realizó la gestión",
    "información facilitada",
    "consulta registrada",
    "sin incidencias destacables",
    "proceso estándar"
]

proto_negativo = [
    "problemas con la atención",
    "no se resolvió el problema",
    "incidencias en el servicio",
    "experiencia insatisfactoria",
    "mala gestión del caso"
]

def embedding_medio(frases):
    embs = []
    for f in frases:
        emb = obtener_embedding(f)[0]
        embs.append(normalizar(emb))
    return np.mean(embs, axis=0)

emb_pos = embedding_medio(proto_positivo)
emb_neu = embedding_medio(proto_neutro)
emb_neg = embedding_medio(proto_negativo)

df_neutros = df[df["sentimiento"] == 2].copy()

def reclasificar_neutro(texto, delta=0.08):
    emb_texto = obtener_embedding(texto)[0]
    emb_texto = normalizar(emb_texto)

    sim_pos = cosine_similarity([emb_texto], [emb_pos])[0][0]
    sim_neu = cosine_similarity([emb_texto], [emb_neu])[0][0]
    sim_neg = cosine_similarity([emb_texto], [emb_neg])[0][0]

    sims = {
        "Positivo_leve": sim_pos,
        "Neutro_real": sim_neu,
        "Negativo_leve": sim_neg
    }

    etiqueta_max = max(sims, key=sims.get)

    
    sims_sorted = sorted(sims.values(), reverse=True)
    if sims_sorted[0] - sims_sorted[1] < delta:
        return "Neutro_real", sims_sorted[0]

    return etiqueta_max, sims[etiqueta_max]

df_neutros = df[df["sentimiento"] == 2].copy()

df_neutros["sub_sentimiento"], df_neutros["sub_confianza"] = zip(
    *df_neutros["texto"].apply(reclasificar_neutro)
)

df.loc[df_neutros.index, "sub_sentimiento"] = df_neutros["sub_sentimiento"]
df.loc[df_neutros.index, "sub_confianza"] = df_neutros["sub_confianza"]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 396.38it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: C:\Users\GGNADI2\OneDrive - MAPFRE\Escritorio\Análisis_sentimiento\beto-sentiment
Key                          | Status     |  | 
-----------------------------+------------+--+-
classifier.weight            | UNEXPECTED |  | 
classifier.bias              | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
BertModel LOAD REPORT from: C:\Users\GGNADI2\OneDrive - MAPFRE\Escritorio\Análisis_sentimiento\beto-sentiment
Key                          | Status     |  | 
-----------------------------+------------+--+-
classifier.weight            | UNEXPECTED |  | 
classifier.bias              | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/

Segunda vuelta análisis (había demasiados comentarios mal clasificados como neutros)

In [5]:
import pandas as pd
import numpy as np

def sentimiento_final(row):
    base = row.get("sentimiento_label", np.nan)
    sub = row.get("sub_sentimiento", np.nan)

    
    if base in ["Positivo", "Negativo"]:
        return base

    
    if sub == "Positivo_leve":
        return "Positivo"
    elif sub == "Negativo_leve":
        return "Negativo"
    else:
        return "Neutro"

df["sentimiento"] = df.apply(sentimiento_final, axis=1)

df["sentimiento"] = df["sentimiento"].fillna("Neutro")


Análisis de aspectos

In [ ]:
from collections import defaultdict

import pandas as pd
from transformers import pipeline

RUTA_MODELO = r"C:\Users\GGNADI2\OneDrive - MAPFRE\Escritorio\Análisis_sentimiento\beto-sentiment"

aspect_model = pipeline(
    "sentiment-analysis",
    model=RUTA_MODELO,
    tokenizer=RUTA_MODELO,
    local_files_only=True,
    truncation=True,
    max_length=512,
    device=0 if torch.cuda.is_available() else -1,
)


aspectos = {

    'Atencion_y_Trato': [
        'trato', 'buen trato', 'mal trato',
        'amabilidad', 'amable', 'desagradable', 'antipático',
        'empatía', 'empatia', 'falta de empatía',
        'atención', 'buena atención', 'mala atención',
        'atención deficiente',
        'profesionalidad', 'profesional', 'poco profesional',
        'explicaciones', 'explicación', 'explican bien', 'no explican',
        'operador', 'teleoperador'
    ],

    'Solicitud_de_Asistencia': [
        'llamada', 'llamar', 'solicitar asistencia',
        'petición', 'aviso',
        'no cogen', 'no responden',
        'espera al teléfono',
        'repetir llamada',
        'apertura de incidencia'
    ],

    'Servicio_de_Grua': [
        'grúa', 'grua',
        'no viene la grúa', 'no llegó la grúa',
        'plantón',
        'espera de grúa', 'tiempo de grúa',
        'remolque',
        'enganchar el coche',
        'cargar el coche'
    ],

    'Tiempo_de_Asistencia': [
        'espera', 'tiempo de espera',
        'mucho tiempo', 'larga espera',
        'demora', 'retraso', 'retrasos', 'tardanza',
        'horas', 'hora', 'más de una hora',
        'rápido', 'rapidez',
        'lento', 'lentitud',
        'urgente', 'urgencia'
    ],

    'Tipo_de_Averia': [
        'avería', 'averia',
        'pinchazo', 'rueda',
        'batería', 'bateria',
        'no arranca',
        'coche parado',
        'fallo mecánico',
        'combustible', 'sin gasolina'
    ],

    'Traslado_y_Destino': [
        'traslado', 'llevar el coche',
        'destino',
        'taller', 'taller cercano',
        'domicilio',
        'distancia', 'kilómetros',
        'kilometraje',
        'no lo llevan donde quiero'
    ],

    'Cobertura_de_Asistencia': [
        'cobertura', 'no cubre', 'no lo cubre',
        'cobertura insuficiente',
        'condiciones',
        'poliza',
        'kilómetros incluidos',
        'limitación',
        'exclusiones'
    ],

    'Coste_y_Pagos': [
        'pago', 'pagar',
        'coste', 'coste extra',
        'cobro', 'me cobraron',
        'factura',
        'precio',
        'no incluido'
    ],

    'Canales_y_Comunicacion': [
        'teléfono', 'contacto',
        'dificultad para contactar',
        'comunicación', 'mala comunicación',
        'respuesta', 'sin respuesta',
        'robot', 'contestador',
        'app', 'aplicación',
        'mensaje', 'sms'
    ]
}

def find_aspects(text, aspectos):
    text_lower = text.lower()
    return [category for category, keywords in aspectos.items() if any(keyword in text_lower for keyword in keywords)]


df["texto"] = df["ES - Corredores - Verbatim Comment including Corredores"].astype(str)

df["aspect_matches"] = df["texto"].apply(lambda t: find_aspects(t, aspectos))

rows = []
for idx, aspects_list in df["aspect_matches"].items():
    text = df.at[idx, "texto"]
    for aspect in aspects_list:
        rows.append({
            "idx": idx,
            "aspect": aspect,
            "input_text": f"{aspect}. {text}"
        })

predictions = []
rows_texts = [row["input_text"] for row in rows]
for i in range(0, len(rows_texts), batch_size):
    batch_texts = rows_texts[i:i + batch_size]
    predictions.extend(aspect_model(batch_texts))

results = defaultdict(list)
for row, pred in zip(rows, predictions):
    results[row["idx"]].append(
        str(row["aspect"])
    )


df["aspectos_absa"] = df.index.map(lambda idx: results.get(idx, []))

df = df.drop(columns=["aspect_matches"], errors="ignore")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 367.13it/s, Materializing param=classifier.weight]                        
BertForSequenceClassification LOAD REPORT from: C:\Users\GGNADI2\OneDrive - MAPFRE\Escritorio\Análisis_sentimiento\beto-sentiment
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
BertForSequenceClassification LOAD REPORT from: C:\Users\GGNADI2\OneDrive - MAPFRE\Escritorio\Análisis_sentimiento\beto-sentiment
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Creación columna aspectos + test y train

In [ ]:
from sklearn.model_selection import train_test_split

def find_aspects(text, aspectos):
    text_lower = text.lower()
    return [category for category, keywords in aspectos.items() if any(keyword in text_lower for keyword in keywords)]


def apply_absa(data, aspectos, aspect_model, batch_size=32):
    data = data.copy()
    data["texto"] = data["ES - Corredores - Verbatim Comment including Corredores"].astype(str)
    data["aspect_matches"] = data["texto"].apply(lambda t: find_aspects(t, aspectos))

    rows = []
    for idx, aspects_list in data["aspect_matches"].items():
        text = data.at[idx, "texto"]
        for aspect in aspects_list:
            rows.append({
                "idx": idx,
                "aspect": aspect,
                "input_text": f"{aspect}. {text}"
            })

    predictions = []
    rows_texts = [row["input_text"] for row in rows]
    for i in range(0, len(rows_texts), batch_size):
        batch_texts = rows_texts[i:i + batch_size]
        predictions.extend(aspect_model(batch_texts))

    results = defaultdict(list)
    for row, pred in zip(rows, predictions):
        results[row["idx"]].append(str(row["aspect"]))

    data["aspectos_absa"] = data.index.map(lambda idx: results.get(idx, []))
    data = data.drop(columns=["aspect_matches"], errors="ignore")
    return data

# 70/30 train/test split antes de ABSA
train_ratio_70 = 0.7
train_70, test_30 = train_test_split(
    df,
    train_size=train_ratio_70,
    test_size=1 - train_ratio_70,
    random_state=42,
    stratify=df["sentimiento"] if "sentimiento" in df.columns else None
)

# 30/70 train/test split antes de ABSA
train_ratio_30 = 0.3
train_30, test_70 = train_test_split(
    df,
    train_size=train_ratio_30,
    test_size=1 - train_ratio_30,
    random_state=42,
    stratify=df["sentimiento"] if "sentimiento" in df.columns else None
)

print("Split shapes antes de ABSA:")
print("70/30 train:", train_70.shape)
print("70/30 test:", test_30.shape)
print("30/70 train:", train_30.shape)
print("30/70 test:", test_70.shape)

train_70 = apply_absa(train_70, aspectos, aspect_model)
test_30 = apply_absa(test_30, aspectos, aspect_model)
train_30 = apply_absa(train_30, aspectos, aspect_model)
test_70 = apply_absa(test_70, aspectos, aspect_model)
df = apply_absa(df, aspectos, aspect_model)

for data_name, data in [
    ("train_70", train_70),
    ("test_30", test_30),
    ("train_30", train_30),
    ("test_70", test_70),
    ("full", df)
]:
    for aspecto_categoria in aspectos.keys():
        columna = f"aspecto_{aspecto_categoria}"
        data[columna] = data["aspectos_absa"].apply(
            lambda lista_aspectos: int(
                isinstance(lista_aspectos, list) and any(
                    (
                        isinstance(aspecto, dict) and aspecto.get("aspect") == aspecto_categoria
                    ) or (
                        isinstance(aspecto, str) and aspecto == aspecto_categoria
                    )
                    for aspecto in lista_aspectos
                )
            )
        )
    data["aspectos_count"] = data["aspectos_absa"].apply(
        lambda x: len(x) if isinstance(x, list) else 0
    )

train_70.to_excel("asis_train_70_30.xlsx", index=False)
test_30.to_excel("asis_test_70_30.xlsx", index=False)
train_30.to_excel("asis_train_30_70.xlsx", index=False)
test_70.to_excel("asis_test_30_70.xlsx", index=False)

df.to_excel("asis_completo_analizado_absa.xlsx", index=False)
